In [1]:
import pandas as pd
import numpy as np
import spacy
import re
import ast

from collections import defaultdict

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

In [2]:
sent_df = pd.read_csv("../data/processed/segmented_sentences.csv")

print("Total sentences:", len(sent_df))
print("Unique articles:", sent_df["article_id"].nunique())

sent_df.head()

Total sentences: 25
Unique articles: 3


,article_id,publication_date,sentence_id,sentence
0,article_01,2026-08-14,1,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning."
1,article_01,2026-08-14,2,The city weather office recorded 118 millimetres of rain in twelve hours.
2,article_01,2026-08-14,3,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.
3,article_01,2026-08-14,4,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge."
4,article_01,2026-08-14,5,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.


In [15]:
# Reporting verbs that generally introduce statements
# rather than describe the actual news event.

REPORTING_VERBS = {
    "say", "report", "state", "tell",
    "add", "note", "write", "mention",
    "claim", "explain", "announce"
}


def classify_predicate(token):

    if token.pos_ != "VERB":
        return None

    lemma = token.lemma_.lower()

    if lemma in REPORTING_VERBS:
        return None

    if token.dep_ == "ROOT":
        return "main"

    if token.dep_ == "conj":
        return "coordinated"

    if token.dep_ in ["advcl", "ccomp", "xcomp"]:
        return "subordinate"

    if token.dep_ in ["acl", "relcl"]:
        if token.tag_ in ["VBG", "VBN"]:
            return "participial"

    return None

In [16]:
event_candidates = []

for _, row in sent_df.iterrows():

    doc = nlp(row["sentence"])

    for token in doc:

        predicate_type = classify_predicate(token)

        if predicate_type is None:
            continue

        event_candidates.append({

            "article_id": row["article_id"],
            "sentence_id": row["sentence_id"],
            "publication_date": row["publication_date"],

            "original_sentence": row["sentence"],

            "event_trigger": token.text,
            "action": token.lemma_,

            "subject": [
                child.text for child in token.children
                if child.dep_ in [
                    "nsubj", "nsubjpass", "nsubj:pass"
                ]
            ],

            "dependency": token.dep_,
            "head_action": token.head.lemma_,

            "predicate_type": predicate_type,

            "pos_tag": token.tag_

        })


event_candidates_df = pd.DataFrame(event_candidates)

print("Total potential events:", len(event_candidates_df))

display(event_candidates_df.head(25))

Total potential events: 42


,article_id,sentence_id,publication_date,original_sentence,event_trigger,action,subject,dependency,head_action,predicate_type,pos_tag
0,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",began,begin,[rainfall],ROOT,begin,main,VBD
1,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",causing,cause,[],advcl,begin,subordinate,VBG
2,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",flood,flood,[streets],ccomp,cause,subordinate,VB
3,article_01,2,2026-08-14,The city weather office recorded 118 millimetres of rain in twelve hours.,recorded,record,[office],ROOT,record,main,VBD
4,article_01,3,2026-08-14,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.,received,receive,[services],ROOT,receive,main,VBD
5,article_01,4,2026-08-14,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",entered,enter,[Water],ROOT,enter,main,VBD
6,article_01,4,2026-08-14,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",diverted,divert,[traffic],conj,enter,coordinated,VBN
7,article_01,5,2026-08-14,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,opened,open,[corporation],ROOT,open,main,VBD
8,article_01,6,2026-08-14,Officials said that 46 residents had been moved to the shelters by 10 a.m.,moved,move,[residents],ccomp,say,subordinate,VBN
9,article_01,7,2026-08-14,The weather office warned that further rainfall was expected through Saturday.,warned,warn,[office],ROOT,warn,main,VBD


In [17]:
SUBJECT_DEPS = {
    "nsubj",
    "nsubjpass",
    "nsubj:pass",
    "csubj",
    "csubjpass"
}

OBJECT_DEPS = {
    "dobj",
    "obj",
    "iobj",
    "dative"
}


def get_noun_phrase(token):

    for chunk in token.doc.noun_chunks:

        if chunk.root == token:

            phrase_tokens = [
                t for t in chunk
                if not (
                    t.dep_ == "amod"
                    and t.lemma_.lower() == "lead"
                    and t.tag_ == "VBG"
                )
            ]

            return " ".join(t.text for t in phrase_tokens).strip()

    return token.text.strip()


def resolve_subject(token):

    # Case 1: Direct subject
    direct_subjects = [
        child for child in token.children
        if child.dep_ in SUBJECT_DEPS
    ]

    if direct_subjects:

        return [
            get_noun_phrase(child)
            for child in direct_subjects
        ], "direct"

    # Case 2: Coordinated predicate
    if token.dep_ == "conj":

        inherited, source = resolve_subject(token.head)

        if inherited:
            return inherited, "inherited_coordinated"

    # Case 3: Participial and adverbial constructions
    if token.dep_ in ["advcl", "amod", "acl"]:

        inherited, source = resolve_subject(token.head)

        if inherited:
            return inherited, "inherited_subordinate"

    # Case 4: Object-controlled infinitive
    if token.dep_ == "xcomp":

        controlling_objects = [
            child for child in token.head.children
            if child.dep_ in OBJECT_DEPS
        ]

        if controlling_objects:

            return [
                get_noun_phrase(child)
                for child in controlling_objects
            ], "inferred_from_object"

        # Subject-controlled infinitive
        inherited, source = resolve_subject(token.head)

        if inherited:
            return inherited, "inherited_from_head"

    return [], "missing"

In [18]:
def extract_objects(token):

    objects = []

    object_deps = {
        "dobj",
        "obj",
        "iobj",
        "dative",
        "attr",
        "oprd"
    }

    for child in token.children:

        if child.dep_ in object_deps:

            objects.append(get_noun_phrase(child))

    return list(dict.fromkeys(objects))

In [19]:
def extract_agents(token):

    agents = []

    for child in token.children:

        if child.dep_ in ["agent", "obl:agent"]:

            for agent_child in child.children:

                if agent_child.dep_ == "pobj":

                    agents.append(
                        get_noun_phrase(agent_child)
                    )

    return list(dict.fromkeys(agents))


def is_passive(token):

    if any(
        child.dep_ in ["nsubjpass", "nsubj:pass"]
        for child in token.children
    ):
        return True

    # Additional morphological check
    return "Pass" in token.morph.get("Voice")

In [20]:
def extract_negation(token):

    negation_words = {
        "not", "n't", "never", "no"
    }

    for child in token.children:

        if (
            child.dep_ == "neg"
            or child.text.lower() in negation_words
        ):
            return True

    return False

In [21]:
event_arguments = []

for _, row in event_candidates_df.iterrows():

    sentence = row["original_sentence"]

    doc = nlp(sentence)

    # Locate the original predicate
    token = next(
        (
            t for t in doc
            if t.text == row["event_trigger"]
            and t.lemma_ == row["action"]
        ),
        None
    )

    if token is None:
        continue

    subjects, subject_source = resolve_subject(token)

    objects = extract_objects(token)

    agents = extract_agents(token)

    passive = is_passive(token)

    negation = extract_negation(token)

    event_arguments.append({

        **row.to_dict(),

        "subject": subjects,

        "subject_source": subject_source,

        "object": objects,

        "agent": agents,

        "passive": passive,

        "negation": negation

    })


event_arguments_df = pd.DataFrame(event_arguments)

print("Total extracted event records:", len(event_arguments_df))

display(
    event_arguments_df[
        [
            "original_sentence",
            "event_trigger",
            "subject",
            "subject_source",
            "object",
            "agent",
            "passive",
            "negation"
        ]
    ].head(25)
)

Total extracted event records: 42


,original_sentence,event_trigger,subject,subject_source,object,agent,passive,negation
0,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",began,[Continuous rainfall],direct,[],[],False,False
1,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",causing,[Continuous rainfall],inherited_subordinate,[],[],False,False
2,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",flood,[several streets],direct,[],[],False,False
3,The city weather office recorded 118 millimetres of rain in twelve hours.,recorded,[The city weather office],direct,[118 millimetres],[],False,False
4,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.,received,[Emergency services],direct,[their first major flooding calls],[],False,False
5,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",entered,[Water],direct,[ground - floor homes],[],False,False
6,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",diverted,[traffic],direct,[],[],True,False
7,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,opened,[The municipal corporation],direct,[two temporary shelters],[],False,False
8,Officials said that 46 residents had been moved to the shelters by 10 a.m.,moved,[46 residents],direct,[],[],True,False
9,The weather office warned that further rainfall was expected through Saturday.,warned,[The weather office],direct,[],[],False,False


In [29]:
import re

def extract_temporal_entities(text):

    doc = nlp(text)

    temporal_entities = set()

    # 1. Extract DATE and TIME entities
    for ent in doc.ents:
        if ent.label_ in ["DATE", "TIME"]:
            temporal_entities.add(ent.text.strip())

    # 2. Clock times
    time_pattern = (
        r'\b(?:approximately\s+|around\s+|at\s+)?'
        r'\d{1,2}(?::\d{2})?\s*'
        r'(?:a\.m\.|p\.m\.|am|pm)\b'
    )

    temporal_entities.update(
        re.findall(time_pattern, text, flags=re.IGNORECASE)
    )

    # 3. Days and relative expressions
    relative_patterns = [
        r'\b(?:early|late)\s+(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\s+(?:morning|afternoon|evening)\b',
        r'\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b',
        r'\b(?:today|yesterday|tomorrow|tonight|overnight|noon|midnight)\b',
        r'\b(?:early|late)\s+(?:morning|afternoon|evening)\b',
        r'\b(?:through|until|by)\s+(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b'
    ]

    for pattern in relative_patterns:
        temporal_entities.update(
            re.findall(pattern, text, flags=re.IGNORECASE)
        )

    # 4. Durations
    duration_pattern = (
        r'\b(?:\d+|one|two|three|four|five|six|seven|eight|'
        r'nine|ten|eleven|twelve)\s+'
        r'(?:seconds?|minutes?|hours?|days?|weeks?|months?|years?)\b'
    )

    temporal_entities.update(
        re.findall(duration_pattern, text, flags=re.IGNORECASE)
    )

    # 5. Remove whitespace and duplicates
    temporal_entities = {
        item.strip() for item in temporal_entities if item.strip()
    }

    # 6. Remove shorter expressions contained in longer expressions
    cleaned = []

    for item in temporal_entities:

        is_substring = any(
            item.lower() in other.lower()
            and item.lower() != other.lower()
            for other in temporal_entities
        )

        if not is_substring:
            cleaned.append(item)

    return sorted(cleaned, key=str.lower)


event_arguments_df["temporal_entities"] = (
    event_arguments_df["original_sentence"]
    .apply(extract_temporal_entities)
)

display(
    event_arguments_df[
        ["original_sentence", "temporal_entities"]
    ].head(25)
)


,original_sentence,temporal_entities
0,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.","[August 13, early Friday morning]"
1,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.","[August 13, early Friday morning]"
2,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.","[August 13, early Friday morning]"
3,The city weather office recorded 118 millimetres of rain in twelve hours.,[twelve hours]
4,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.,"[approximately 5:30 a.m., August 14]"
5,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",[]
6,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.",[]
7,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,[]
8,Officials said that 46 residents had been moved to the shelters by 10 a.m.,[10 a.m.]
9,The weather office warned that further rainfall was expected through Saturday.,[through Saturday]


In [30]:
import re

def extract_location_entities(text):

    doc = nlp(text)

    locations = set()

    # 1. Named entity recognition
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC", "FAC"]:
            locations.add(ent.text.strip())

    # 2. Dependency-based location extraction
    location_prepositions = {
        "in", "at", "near", "across",
        "toward", "towards", "from",
        "inside", "outside", "along"
    }

    for token in doc:

        if token.text.lower() not in location_prepositions:
            continue

        for child in token.children:

            if child.dep_ != "pobj":
                continue

            # Extract noun phrase
            phrase_tokens = list(child.subtree)

            phrase = doc[
                min(t.i for t in phrase_tokens):
                max(t.i for t in phrase_tokens) + 1
            ].text.strip()

            # Reject generic references
            generic_phrases = {
                "the river",
                "the two locations",
                "the city",
                "the area"
            }

            if phrase.lower() not in generic_phrases:
                locations.add(phrase)

    # 3. Remove temporal expressions
    temporal_entities = extract_temporal_entities(text)

    temporal_lower = {
        item.lower() for item in temporal_entities
    }

    locations = {
        loc for loc in locations
        if loc.lower() not in temporal_lower
    }

    # 4. Remove duplicates and empty values
    return sorted(
        {loc.strip() for loc in locations if loc.strip()}
    )


event_arguments_df["location_entities"] = (
    event_arguments_df["original_sentence"]
    .apply(extract_location_entities)
)

display(
    event_arguments_df[
        ["original_sentence", "location_entities"]
    ].head(25)
)


,original_sentence,location_entities
0,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.","[Riverton, the eastern part of the city]"
1,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.","[Riverton, the eastern part of the city]"
2,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.","[Riverton, the eastern part of the city]"
3,The city weather office recorded 118 millimetres of rain in twelve hours.,[]
4,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.,[]
5,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.","[the Old Bridge, the Riverside neighbourhood]"
6,"Water entered ground-floor homes in the Riverside neighbourhood, and traffic was diverted from the Old Bridge.","[the Old Bridge, the Riverside neighbourhood]"
7,The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.,[Central High School and the Riverside Community Hall]
8,Officials said that 46 residents had been moved to the shelters by 10 a.m.,[]
9,The weather office warned that further rainfall was expected through Saturday.,[]


In [31]:
assert len(event_candidates_df) == len(event_arguments_df)

assert event_candidates_df["original_sentence"].reset_index(drop=True).equals(
    event_arguments_df["original_sentence"].reset_index(drop=True)
)

assert event_candidates_df["event_trigger"].reset_index(drop=True).equals(
    event_arguments_df["event_trigger"].reset_index(drop=True)
)

print("DataFrames are aligned. Ready to combine.")

DataFrames are aligned. Ready to combine.


In [32]:
final_events_df = event_candidates_df.reset_index(drop=True).copy()

columns_to_add = [
    "subject",
    "subject_source",
    "object",
    "agent",
    "passive",
    "negation",
    "temporal_entities",
    "location_entities",
    "causal_indicators",
    "quantities"
]

for col in columns_to_add:
    if col in event_arguments_df.columns:
        final_events_df[col] = event_arguments_df[col].reset_index(drop=True)

display(final_events_df.head())

,article_id,sentence_id,publication_date,original_sentence,event_trigger,action,subject,dependency,head_action,predicate_type,pos_tag,subject_source,object,agent,passive,negation,temporal_entities,location_entities
0,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",began,begin,[Continuous rainfall],ROOT,begin,main,VBD,direct,[],[],False,False,"[August 13, early Friday morning]","[Riverton, the eastern part of the city]"
1,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",causing,cause,[Continuous rainfall],advcl,begin,subordinate,VBG,inherited_subordinate,[],[],False,False,"[August 13, early Friday morning]","[Riverton, the eastern part of the city]"
2,article_01,1,2026-08-14,"Continuous rainfall began across Riverton late on August 13, causing several streets in the eastern part of the city to flood by early Friday morning.",flood,flood,[several streets],ccomp,cause,subordinate,VB,direct,[],[],False,False,"[August 13, early Friday morning]","[Riverton, the eastern part of the city]"
3,article_01,2,2026-08-14,The city weather office recorded 118 millimetres of rain in twelve hours.,recorded,record,[The city weather office],ROOT,record,main,VBD,direct,[118 millimetres],[],False,False,[twelve hours],[]
4,article_01,3,2026-08-14,Emergency services received their first major flooding calls at approximately 5:30 a.m. on August 14.,received,receive,[Emergency services],ROOT,receive,main,VBD,direct,[their first major flooding calls],[],False,False,"[approximately 5:30 a.m., August 14]",[]


In [33]:
import os

output_dir = "../data/processed/"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "extracted_events_v2.csv")

final_events_df.to_csv(output_path, index=False)

print(f"Saved {len(final_events_df)} extracted events to {output_path}")

Saved 42 extracted events to ../data/processed/extracted_events_v2.csv


Event ID: 7
Sentence: The municipal corporation opened two temporary shelters at Central High School and the Riverside Community Hall.


KeyError: 'action_norm'